<a href="https://colab.research.google.com/github/samanvisingh/whatsapp_chat_analyzer-/blob/main/Minor2_project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
# reading dataset
df = pd.read_csv("rahul_transactions.csv")

In [3]:
# duplicate the rows
df = df.drop_duplicates()

In [4]:
# checking column names , data types and the first few rows
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 1310 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1310 non-null   object 
 1   Time         1310 non-null   object 
 2   Description  1310 non-null   object 
 3   Type         1310 non-null   object 
 4   Amount       1310 non-null   object 
 5   Balance      1310 non-null   float64
 6   Mode         1310 non-null   object 
 7   Ref          1310 non-null   object 
dtypes: float64(1), object(7)
memory usage: 92.1+ KB
None
          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLI

In [5]:
# reload dataset to fix corrupted data
df = pd.read_csv("rahul_transactions.csv").drop_duplicates()

# clean the Amount column properly
df['Amount'] = df['Amount'].astype(str).str.replace('₹','',regex=False)\
                                     .str.replace('Rs.','',regex=False)\
                                     .str.replace(',','',regex=False)\
                                     .str.strip()
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# making type column to lowercase 'debit' and 'credit'
df['Type'] = df['Type'].str.lower().replace({'dr':'debit','cr':'credit'})
print(df.info())
print(df[['Date', 'Amount', 'Type']].head())

<class 'pandas.core.frame.DataFrame'>
Index: 1310 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1310 non-null   object 
 1   Time         1310 non-null   object 
 2   Description  1310 non-null   object 
 3   Type         1310 non-null   object 
 4   Amount       1310 non-null   float64
 5   Balance      1310 non-null   float64
 6   Mode         1310 non-null   object 
 7   Ref          1310 non-null   object 
dtypes: float64(2), object(6)
memory usage: 92.1+ KB
None
          Date   Amount    Type
0   2024-01-01   2462.0   debit
1    01-Jan-24     50.0   debit
2    01-Jan-24  84728.0  credit
3   2024-01-01   1828.0   debit
4  01 Jan 2024    270.0   debit


In [6]:
# printing the sample of the raw descriptions
print(df['Description'].unique()[:30])
vendor_dict ={
    'Swiggy':['SWIGGY','BUNDL'],
    'Zomato':['ZOMATO'],
    'Zepto':['ZEPTO'],
    'Amazon':['AMAZON'],
    'Zerodha':['ZERODHA'],
    'Myntra':['MYNTRA'],
    'Cash Withdrawl':['ATM-WDL'],
}

['AMAZON SELLER SVCS' 'BHIM-BMTC' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'UPI-AMAN-8934@OKAXIS' 'BHIM-BLINKIT' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'UPI-GROWWPAY@HDFCBANK'
 'OLA ELECTRIC' 'BMS MOVIE TICKETS' 'POS OLA-PRIME' 'SWIGGY-INSTAMART'
 'UPI-STARBUCKS@AXIS' 'UPI-THIRDWAVE@OKAXIS' 'ANI Technologies'
 'BMTC BUS PASS' 'POS TRUFFLES' 'FLIPKART INDIA' 'POS SWIGGY-RESTAURANT'
 'GROFERS INDIA P L' 'POS UBER BANGALORE' 'BANGALORE ELEC SUPPLY'
 'TWC INDIA' 'UPI-BESCOM-BILL@HDFCBANK' 'UPI-AMAN-0816@OKAXIS'
 'ROPPEN TRANSPORTATION' 'OLA CABS' 'POS ZOMATO' 'UPI-AMAZONPAY@HDFCBANK']


In [7]:
# to check description against our dictionary
def get_clean_vendor(desc):
  desc = str(desc).upper()
  for clean_name , keywords in vendor_dict.items():
    for kw in keywords:
      if kw in desc:
        return clean_name
  if 'UPI-' in desc or '@' in desc or 'BHIM' in desc:
    return 'Personal Transfer'
  return 'Uncategorised'

In [8]:
category_dict ={
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zepto':'Quick Commerce',
    'Amazon': 'E-commerce',
    'Myntra':'E-commerce',
    'Zerodha':'Investments',
    'Cash Withdrawl':'Cash with',
    'Personal Transfer':'Personal Transfer'
}
# Create the missing vendor_clean column using get_clean_vendor
df['vendor_clean'] = df['Description'].apply(get_clean_vendor)

df['category'] = df['vendor_clean'].map(category_dict).fillna('Uncategorised')

# 2. Calculate the financial metrics
total_credits = df[df['Type'] == 'credit']['Amount'].sum()
total_debits = df[df['Type'] == 'debit']['Amount'].sum()
net_change = total_credits - total_debits
savings_rate = (net_change / total_credits) * 100 if total_credits != 0 else 0

# 3. Group by the newly created 'category' and 'vendor_clean' columns
top_categories = df[df['Type'] == 'debit'].groupby('category')['Amount'].sum().sort_values(ascending=False).head(5)
top_vendors = df[df['Type'] == 'debit'].groupby('vendor_clean')['Amount'].sum().sort_values(ascending=False).head(5)

# 4. Print the results
print(f"Total credits: ₹{total_credits:,.0f}")
print(f"Total debits: ₹{total_debits:,.0f}")
print(f"Savings rate: {savings_rate:.1f}%\n")
print("Top 5 Categories by Spend:\n", top_categories, "\n")
print("Top 5 Vendors by Spend:\n", top_vendors)

Total credits: ₹509,774
Total debits: ₹1,678,901
Savings rate: -229.3%

Top 5 Categories by Spend:
 category
Uncategorised        634073.0
E-commerce           368991.0
Personal Transfer    241341.0
Investments          210000.0
Food Delivery        150839.0
Name: Amount, dtype: float64 

Top 5 Vendors by Spend:
 vendor_clean
Uncategorised        634073.0
Amazon               299462.0
Personal Transfer    241341.0
Zerodha              210000.0
Swiggy                95523.0
Name: Amount, dtype: float64


In [9]:
print(df.columns)

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref', 'vendor_clean', 'category'],
      dtype='object')


In [10]:
total_credits = df[df['Type'] == 'credit']['Amount'].sum()
total_debits = df[df['Type'] == 'debit']['Amount'].sum()
net_change = total_credits - total_debits
if total_credits !=0:
 savings_rate = (net_change / total_credits) *100
else:
 savings_rate = 0
savings_rate = (
    (net_change / total_credits) * 100 if total_credits != 0 else 0

)

In [11]:
total_credits = df[df['Type'] == 'credit']['Amount'].sum()
total_debits = df[df['Type'] == 'debit']['Amount'].sum()
net_change = total_credits - total_debits
if total_credits != 0:
 savings_rate = (net_change / total_credits) * 100
else:
  savings_rate = 0

In [12]:
df['Month'] = df['Date']

monthly_trend = df[df['Type'] == 'debit'].pivot_table(
    values='Amount',
    index='category',
    columns='Month',
    aggfunc='sum'
).fillna(0)

print("--- MONTHLY TREND ---")
print(monthly_trend.head(10))

df['hour'] = df['Time'].str[:2].astype(int)

time_of_day = df[df['Type'] == 'debit'].groupby(['category', 'hour'])['Amount'].count().unstack().fillna(0)
print("\n--- TIME OF DAY PATTERNS (Transaction Count) ---")
if 'Food Delivery' in time_of_day.index:
    print(time_of_day.loc[['Food Delivery']])

--- MONTHLY TREND ---
Month              01 Apr 2024  01 Jan 2024  01 Jun 2024  01 Mar 2024  \
category                                                                
Cash with                  0.0          0.0          0.0          0.0   
E-commerce                 0.0          0.0          0.0          0.0   
Food Delivery              0.0          0.0        697.0          0.0   
Investments                0.0          0.0          0.0          0.0   
Personal Transfer          0.0        270.0          0.0          0.0   
Quick Commerce             0.0          0.0          0.0        595.0   
Uncategorised            288.0          0.0          0.0        461.0   

Month              01 May 2024  01-Apr-24  01-Jan-24  01-Jun-24  01-May-24  \
category                                                                     
Cash with                  0.0        0.0        0.0        0.0        0.0   
E-commerce                 0.0        0.0        0.0        0.0     2135.0   
Food Del

In [19]:
# calculating mean and standard deviation for each category
debits = df[df['Type'] == 'debit'].copy()
debits['cat_mean'] = debits.groupby('category')['Amount'].transform('mean')
debits['cat_std'] = debits.groupby('category')['Amount'].transform('std')

# calculate Z-scores (using underscores for safer column naming)
debits['z_score'] = (debits['Amount'] - debits['cat_mean']) / debits['cat_std']

# top 2% most unusual high spends
anomalies = debits[debits['z_score'] > 2].sort_values('z_score', ascending=False)

# archetype
archetypes = []
total_debits = debits['Amount'].sum()

# the foodie
food_spend = debits[debits['category'].isin(['Food Delivery', 'Restaurants', 'Cafe'])]['Amount'].sum()
if (food_spend / total_debits) > 0.25:
    archetypes.append(f"THE FOODIE ({(food_spend/total_debits)*100:.1f}% on food)")

# The quick commerce junkie (>15% on Q-com)
qcom_spend = debits[debits['category'] == 'Quick Commerce']['Amount'].sum()
if (qcom_spend / total_debits) > 0.15:
    archetypes.append(f"THE QUICK COMMERCE JUNKIE ({(qcom_spend/total_debits)*100:.1f}% on Q-com)")

# the investor (>15% on Investment)
invest_spend = debits[debits['category'] == 'Investments']['Amount'].sum()
if (invest_spend / total_debits) > 0.15:
    archetypes.append(f"THE INVESTOR ({(invest_spend/total_debits)*100:.1f}% on SIPs)")

# rule: The late-night snacker (>50% of food delivery between 9pm and 2AM)
food_orders = debits[debits['category'] == 'Food Delivery']
late_night_orders = food_orders[(food_orders['hour'] >= 21) | (food_orders['hour'] <= 2)]
if len(food_orders) > 0 and (len(late_night_orders) / len(food_orders)) > 0.50:
    late_pct = (len(late_night_orders) / len(food_orders)) * 100
    archetypes.append(f"THE LATE-NIGHT SNACKER ({late_pct:.1f}% food after 9 PM)")

# rule: the YOLO spenders (savings rate <10%)
if savings_rate < 10:
    archetypes.append(f"THE YOLO SPENDER ({savings_rate:.1f}% savings rate)")

# FINAL PRINTED REPORT
print("===============================")
print(" SpendDNA Report - RAHUL SHARMA")
print(f"6 months - {len(df)} transactions - Jan to Jun 2024")
print("===============================\n")

print("EXECUTIVE SUMMARY")
print(f"  Total credits    : Rs.{total_credits:,.0f}")
print(f"  Total debits     : Rs.{total_debits:,.0f}")
print(f"  Net change       : Rs.{net_change:,.0f}")
print(f"  Savings rate     : {savings_rate:.1f}%\n")

print(" TOP ANOMALIES (Z > 2)")
for _, row in anomalies.head(3).iterrows():
    # parse mixed string formats to safely format the date
    parsed_date = pd.to_datetime(row['Date'], errors='coerce')
    date_str = parsed_date.strftime('%d %b') if pd.notnull(parsed_date) else str(row['Date'])
    print(f"  {date_str:<11} - {row['vendor_clean']:<17} Rs.{row['Amount']:<8.0f} (z={row['z_score']:.1f})")

print("\n RAHUL'S SPENDING ARCHETYPES")
for arch in archetypes:
    print(f"  -> {arch}")
print("\n======================================")

 SpendDNA Report - RAHUL SHARMA
6 months - 1310 transactions - Jan to Jun 2024

EXECUTIVE SUMMARY
  Total credits    : Rs.509,774
  Total debits     : Rs.1,678,901
  Net change       : Rs.-1,169,127
  Savings rate     : -229.3%

 TOP ANOMALIES (Z > 2)
  27 May      - Personal Transfer Rs.17831    (z=7.5)
  03 Mar      - Uncategorised     Rs.18000    (z=7.1)
  04 May      - Uncategorised     Rs.18000    (z=7.1)

 RAHUL'S SPENDING ARCHETYPES
  -> THE YOLO SPENDER (-229.3% savings rate)



In [ ]:
## KEY INSIGHTS
#1. Rahul is burning through his savings with a severely negative savings rate, driven heavily by discretionary spending.
#2. His late-night food delivery habit is a massive cash sink, with over 60% of his Swiggy/Zomato orders happening after 9 PM.
#3. Despite the heavy spending on convenience (Quick Commerce and Food Delivery), he maintains a disciplined 16% allocation to investments via Zerodha SIPs.